# OPC UA — jelajah, baca, subscribe, tulis

OPC UA adalah cara PLC, SCADA, dan MES modern membuka datanya: address space berisi objek, variabel, dan method, dicapai lewat sesi yang aman. IoTCom.Net membungkus stack OPC Foundation; notebook ini menjalankan simulator pabriknya di dalam proses.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.OpcUa, 0.17.0-preview.1"

## Jalankan simulator pabrik dan sambungkan
Client memilih endpoint paling aman (Basic256Sha256, SignAndEncrypt); kedua pihak membuat sertifikat aplikasi self-signed saat pertama dipakai. `ReadOnly = true` sampai kita memutuskan untuk menulis.

In [ ]:
using IoTCom.Net.Adapters.OpcUa;

var probe = new System.Net.Sockets.TcpListener(System.Net.IPAddress.Loopback, 0);
probe.Start();
var uaPort = ((System.Net.IPEndPoint)probe.LocalEndpoint).Port;
probe.Stop();
var plc = OpcUaPlantServer.Create(o => { o.Port = uaPort; o.Interval = TimeSpan.FromMilliseconds(300); });
await plc.StartAsync();
var ua = OpcUaClient.Create(o => { o.UseEndpoint(plc.EndpointUrl); o.AcceptUntrustedCertificates = true; o.ReadOnly = true; });
await ua.ConnectAsync();
Console.WriteLine($"{plc.EndpointUrl} · {ua.SecurityPolicy} / {ua.SecurityMode}");

## Jelajah dan baca
Node id membawa indeks namespace: `ns=2;s=Plant/Line1/Filler/Speed`.

In [ ]:
async Task Tree(string? nodeId, string indent)
{
    foreach (var n in await ua.BrowseAsync(nodeId))
    {
        if (n.NodeId.StartsWith("i=")) continue;
        var value = n.NodeClass == "Variable" ? " = " + (await ua.ReadAsync(n.NodeId)).Text : "";
        Console.WriteLine($"{indent}{n.DisplayName}{value}");
        if (n.IsContainer) await Tree(n.NodeId, indent + "  ");
    }
}
await Tree(null, "");

## Subscribe
Server mengambil sampel monitored item dan hanya menerbitkan perubahan.

In [ ]:
using var window = new CancellationTokenSource(TimeSpan.FromSeconds(2));
var changes = 0;
try
{
    await foreach (var v in ua.SubscribeAsync([plc.NodeId("Line1/Tank7/Level"), plc.NodeId("Line1/Filler/Speed")], TimeSpan.FromMilliseconds(200), window.Token))
        if (changes++ < 8) Console.WriteLine($"{v.SourceTimestamp?.ToLocalTime():HH:mm:ss.fff} {v.NodeId} = {v.Text}");
}
catch (OperationCanceledException) { }
Console.WriteLine($"{changes} notifications in 2 s");

## Penulisan ditolak sampai Anda mengizinkannya

In [ ]:
try { await ua.WriteAsync(plc.NodeId("Line1/Filler/Setpoint"), 100); }
catch (IoTCom.Net.ReadOnlyModeException e) { Console.WriteLine("read-only: " + e.Message); }

await using var operatorUa = OpcUaClient.Create(o => { o.UseEndpoint(plc.EndpointUrl); o.AcceptUntrustedCertificates = true; });
await operatorUa.ConnectAsync();
await operatorUa.WriteAsync(plc.NodeId("Line1/Filler/Setpoint"), "100");   // converted to Double
Console.WriteLine($"setpoint now {(await ua.ReadAsync(plc.NodeId("Line1/Filler/Setpoint"))).Text}");
var previous = await operatorUa.CallAsync(plc.NodeId("Line1"), plc.NodeId("Line1/ResetCounter"));
Console.WriteLine($"ResetCounter() returned {previous[0]}");
await ua.DisposeAsync();
await plc.DisposeAsync();

## Lebih lanjut
`iotcom opcua browse --sim`, `iotcom opcua watch`, *Penjelajah tag OPC UA* di Gallery, dan sampel OpcUaBrowser. Lihat `docs/id/protocols/opcua.md` untuk sertifikat dan trust.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*